# เทรนโมเดลคำปลุก "สายฝน" (openWakeWord) บน Google Colab ฟรี

**อ่านก่อนกดอะไร**

notebook นี้เทรนโมเดลจากไฟล์เสียงที่สร้างไว้แล้วบน VPS ไม่ได้สร้างเสียงใหม่ใน Colab
จึงข้ามขั้น `--generate_clips` ของ openWakeWord ไปเลย

### สิ่งที่ต้องเตรียมมา

ไฟล์ `wake-samples.zip` จาก VPS (ดู `server/INSTALL.md` หัวข้อชุดเสียงเทรนคำปลุก)

### ตั้ง GPU ก่อนเริ่ม

เมนู **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

ถ้าไม่ตั้ง จะเทรนบน CPU และใช้เวลานานหลายชั่วโมง

---

### ⚠️ ข้อควรรู้ที่ต้องบอกก่อน

notebook อัตโนมัติของ openWakeWord ต้นทางมี issue เปิดอยู่ว่าใช้ไม่ได้
([#296](https://github.com/dscripka/openWakeWord/issues/296)) และเรื่องคุณภาพ
([#110](https://github.com/dscripka/openWakeWord/issues/110)) notebook นี้จึง
**ปักรุ่นไว้ที่ v0.6.0** และลงแพ็กเกจที่ issue บอกว่าขาด (`onnx`) ไว้ให้แล้ว

**แต่ผมยังไม่ได้รันมันจริง** เพราะไม่มี GPU และไม่มีสิทธิ์รัน Colab
ถ้าเจอ error ให้ส่งข้อความ error มา อย่าพยายามแก้เอง


## ขั้น 1 — ติดตั้ง


In [ ]:
# ปักรุ่นไว้ที่ v0.6.0 ไม่ใช้ main เพราะ main เปลี่ยนแล้ว notebook พัง
!git clone --depth 1 --branch v0.6.0 https://github.com/dscripka/openWakeWord.git /content/openWakeWord
%cd /content/openWakeWord
!pip install -q -e .

# train.py import piper-sample-generator ตั้งแต่ตอนโหลดไฟล์ แม้เราจะไม่ใช้
# ขั้นสร้างเสียงของมัน จึงต้อง clone มาไม่งั้น import พัง
!git clone --depth 1 https://github.com/dscripka/piper-sample-generator.git /content/piper-sample-generator

# onnx ไม่อยู่ใน requirements ของ v0.6.0 (issue #296) ต้องลงเอง
!pip install -q onnx onnxruntime torch-audiomentations audiomentations datasets speechbrain

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ไม่มี — ไปตั้ง Runtime ก่อน')


## ขั้น 2 — อัปโหลด wake-samples.zip

กดรันเซลล์นี้ แล้วกดปุ่ม **Choose Files** ที่โผล่ขึ้นมา เลือกไฟล์ `wake-samples.zip`


In [ ]:
from google.colab import files
import zipfile, pathlib, shutil

uploaded = files.upload()
name = next(iter(uploaded))

raw = pathlib.Path('/content/raw')
shutil.rmtree(raw, ignore_errors=True)
with zipfile.ZipFile(name) as zf:
    zf.extractall(raw)

# zip เก็บไฟล์ไว้ใต้โฟลเดอร์ชั้นเดียว หาให้เจอไม่ว่าชื่ออะไร
root = next(p for p in [raw, *raw.iterdir()] if (p / 'positive').is_dir())
for label in ('positive', 'nearmiss', 'background'):
    print(label, len(list((root / label).glob('*.wav'))), 'ไฟล์')


## ขั้น 3 — จัดไฟล์เข้าโฟลเดอร์ที่ openWakeWord ต้องการ

openWakeWord อ่านจากสี่โฟลเดอร์: `positive_train`, `positive_test`,
`negative_train`, `negative_test`

**แบ่ง 85/15 โดยแยกตามเสียงผู้พูด ไม่ใช่สุ่มทีละไฟล์** — ถ้าสุ่มทีละไฟล์
เสียงเดียวกันจะอยู่ทั้งสองชุด แล้วคะแนนทดสอบจะสวยเกินจริง


In [ ]:
import random, re, shutil, pathlib

out = pathlib.Path('/content/saifon_model')
shutil.rmtree(out, ignore_errors=True)
dirs = {k: out / k for k in ('positive_train','positive_test','negative_train','negative_test')}
for d in dirs.values():
    d.mkdir(parents=True, exist_ok=True)

def voice_of(path):
    # ชื่อไฟล์: <label>_<index>_<Voice>_<rate>.wav
    parts = path.stem.split('_')
    return parts[2] if len(parts) > 2 else 'unknown'

voices = sorted({voice_of(p) for p in (root / 'positive').glob('*.wav')})
random.Random(20260922).shuffle(voices)
held_out = set(voices[: max(1, len(voices) * 15 // 100)])
print('เสียงที่กันไว้เป็นชุดทดสอบ:', sorted(held_out))

counts = {k: 0 for k in dirs}
for label, kind in (('positive','positive'), ('nearmiss','negative'), ('background','negative')):
    for path in (root / label).glob('*.wav'):
        split = 'test' if voice_of(path) in held_out else 'train'
        target = f'{kind}_{split}'
        shutil.copy(path, dirs[target] / path.name)
        counts[target] += 1

for k, v in counts.items():
    print(f'{k:<16} {v}')


## ขั้น 4 — โหลดชุดเสียงรบกวนและเสียงก้องห้อง

การ augment ต้องมีเสียงพื้นหลังจริงกับ room impulse response มาผสม
ขั้นนี้คือที่ที่ความหลากหลายของ **เสียงดัง เสียงก้อง ระดับเสียง และระดับเสียงสูงต่ำ**
ถูกสร้างขึ้นฟรี — เราจึงไม่จ่ายเงินซื้อความหลากหลายพวกนี้จาก Google

ใช้เวลาประมาณ 5–10 นาที


In [ ]:
import os, scipy.io.wavfile, numpy as np, datasets
from pathlib import Path

# Room impulse responses (MIT)
Path('/content/mit_rirs').mkdir(exist_ok=True)
rir = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses',
                            split='train', streaming=True)
for row in rir:
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(f"/content/mit_rirs/{name}", 16000,
                           (row['audio']['array'] * 32767).astype(np.int16))
print('RIR:', len(os.listdir('/content/mit_rirs')))

# เสียงพื้นหลัง และชุด feature สำหรับวัด false positive
!mkdir -p /content/background_clips
!wget -q -O /content/validation_set_features.npy \
  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print('validation features:', os.path.getsize('/content/validation_set_features.npy') // 1024, 'KB')


### เสียงพื้นหลังเพิ่มเติม (ทางเลือก แต่แนะนำ)

ยิ่งเสียงพื้นหลังเหมือนห้องจริงของบ้านเรา อัตราปลุกผิดจะยิ่งวัดได้ตรง
ถ้ามีเวลา ให้โหลดชุด audioset ย่อยด้านล่าง (ประมาณ 10–20 นาที)


In [ ]:
# ถ้าข้ามขั้นนี้ การ augment จะใช้เฉพาะ coloured noise ซึ่งอ่อนกว่าเสียงจริง
import datasets, scipy.io.wavfile, numpy as np

audioset = datasets.load_dataset('agkphysics/AudioSet', split='train', streaming=True)
for i, row in enumerate(audioset):
    if i >= 800:
        break
    audio = row['audio']['array']
    if len(audio) < 16000:
        continue
    scipy.io.wavfile.write(f'/content/background_clips/{i:05d}.wav', 16000,
                           (audio[:16000 * 10] * 32767).astype(np.int16))
import os; print('background clips:', len(os.listdir('/content/background_clips')))


## ขั้น 5 — เขียนไฟล์ config

ค่าที่ต่างจากตัวอย่างต้นทาง และเหตุผล:

| ค่า | ตั้งเป็น | ทำไม |
|---|---|---|
| `target_false_positives_per_hour` | **0.125** | เกณฑ์ของ Poom คือไม่เกิน 1 ครั้งต่อ 8 ชั่วโมง = 0.125 ต่อชั่วโมง |
| `augmentation_rounds` | **25** | เรามีคลิปจริงแค่ 1,050 ไฟล์ การ augment คือสิ่งที่ขยายให้พอเทรน |
| `custom_negative_phrases` | คำใกล้เคียงไทย | คำที่ต้องไม่ปลุก ระบุตรงๆ ดีกว่าให้ระบบเดาจาก phoneme |
| `n_samples` | ไม่ได้ใช้ | ใช้เฉพาะขั้น `--generate_clips` ซึ่งเราข้าม |


In [ ]:
config = '''
model_name: "saifon"

target_phrase:
  - "สายฝน"

custom_negative_phrases:
  - "สายลม"
  - "สายไฟ"
  - "สายด่วน"
  - "สายพาน"
  - "ฝนตก"
  - "ฝนหยุด"
  - "สายฝัน"
  - "ชายฝน"

# ไม่ได้ใช้ เพราะเราข้ามขั้นสร้างเสียง แต่ train.py อ่านคีย์นี้
n_samples: 1000
n_samples_val: 200
tts_batch_size: 50
piper_sample_generator_path: "/content/piper-sample-generator"

output_dir: "/content"

rir_paths:
  - "/content/mit_rirs"
background_paths:
  - "/content/background_clips"
background_paths_duplication_rate:
  - 1

false_positive_validation_data_path: "/content/validation_set_features.npy"

# คลิปจริงมีน้อย จึง augment หลายรอบ ความสุ่มในแต่ละรอบทำให้ได้ตัวอย่างต่างกัน
augmentation_rounds: 25
augmentation_batch_size: 16

feature_data_files:
  "ACAV100M_sample": "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"

batch_n_per_class:
  "ACAV100M_sample": 1024
  "adversarial_negative": 50
  "positive": 50

model_type: "dnn"
layer_size: 32
steps: 20000
max_negative_weight: 1500

# เกณฑ์ของ Poom: ปลุกผิดไม่เกิน 1 ครั้งต่อ 8 ชั่วโมง
target_false_positives_per_hour: 0.125
'''

open('/content/saifon.yml', 'w').write(config)
print(config)


### ชุด feature เสียงลบขนาดใหญ่

การเทรนต้องมีเสียง "ไม่ใช่คำปลุก" จำนวนมหาศาลเป็นฐาน ไฟล์นี้ประมาณ 2,000 ชั่วโมง
**ไฟล์ใหญ่ ใช้เวลาโหลดสักพัก**


In [ ]:
!wget -q --show-progress -O /content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy \
  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
import os; print(os.path.getsize('/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy') / 1e9, 'GB')


## ขั้น 6 — augment แล้วเทรน

**ไม่ใส่ `--generate_clips`** เพราะเสียงมาจาก Google TTS แล้ว

ขั้นนี้นานที่สุด ประมาณ 30–60 นาที ถ้า Colab ตัดกลางทางให้รันเซลล์นี้ซ้ำ
(`--augment_clips` จะข้ามงานที่ทำไว้แล้วถ้าไม่ใส่ `--overwrite`)


In [ ]:
%cd /content/openWakeWord
!python -m openwakeword.train --training_config /content/saifon.yml --augment_clips --train_model


## ขั้น 7 — ดูผลวัดเบื้องต้น

🔴 **ตัวเลขจาก Colab ไม่ใช่การผ่านเกณฑ์** อ่านคำเตือนใต้ตารางด้วย


In [ ]:
import numpy as np, onnxruntime as ort, pathlib, glob

model_paths = glob.glob('/content/saifon/*.onnx') + glob.glob('/content/**/saifon*.onnx', recursive=True)
print('ไฟล์โมเดลที่หาได้:', model_paths)

pos = np.load('/content/saifon/positive_features_test.npy')
neg = np.load('/content/saifon/negative_features_test.npy')
print('ชุดทดสอบ: positive', pos.shape, 'negative', neg.shape)

session = ort.InferenceSession(model_paths[0])
name = session.get_inputs()[0].name

def scores(features):
    return np.concatenate([session.run(None, {name: features[i:i+256].astype(np.float32)})[0].flatten()
                           for i in range(0, len(features), 256)])

pos_scores, neg_scores = scores(pos), scores(neg)

print()
print(f'{"threshold":>10} {"hit rate":>10} {"false alarms":>14}')
for threshold in (0.3, 0.5, 0.7, 0.9, 0.95):
    hit = (pos_scores >= threshold).mean() * 100
    false = (neg_scores >= threshold).mean() * 100
    print(f'{threshold:>10.2f} {hit:>9.1f}% {false:>13.2f}%')

print()
print('เลือก threshold ที่ hit rate ยังสูงและ false alarms ต่ำที่สุด แล้วจดไว้')


> ## 🔴 ตัวเลขข้างบนยังไม่ใช่การผ่านเกณฑ์
>
> เกณฑ์ที่ Poom ตั้งไว้คือ **แม่น ≥90% ที่ระยะ 3 เมตร** และ
> **ปลุกผิดไม่เกิน 1 ครั้งต่อ 8 ชั่วโมง** ทั้งสองข้อวัดได้บน **Samsung Galaxy A07 เครื่องจริง**
> เท่านั้น ด้วยไมโครโฟนจริง ห้องจริง และเสียงรบกวนจริง
>
> ตัวเลขใน Colab วัดจากเสียงสังเคราะห์ที่ผ่าน augmentation ซึ่ง:
> - ไม่มีลักษณะไมโครโฟนของ A07
> - ไม่มีเสียงสะท้อนของห้องจริง
> - ไม่มีเสียงคนในบ้านจริง
> - เป็นเสียงสังเคราะห์ ไม่ใช่เสียงคนพูด
>
> **ถือว่าตัวเลขนี้เป็นแค่สัญญาณว่าคุ้มค่าจะเอาลงเครื่องไปวัดต่อหรือไม่**
> ขั้นตอนวัดจริงอยู่ใน `WAKEWORD.md`


## ขั้น 8 — ดาวน์โหลดโมเดล


In [ ]:
from google.colab import files
import glob, shutil

onnx = sorted(glob.glob('/content/**/saifon*.onnx', recursive=True))[0]
shutil.copy(onnx, '/content/wakeword.onnx')
print('ขนาด:', __import__('os').path.getsize('/content/wakeword.onnx') / 1024, 'KB')
files.download('/content/wakeword.onnx')

# tflite ด้วย ถ้ามี — เผื่อเลือกใช้ TensorFlow Lite แทน ONNX Runtime ภายหลัง
for tflite in glob.glob('/content/**/saifon*.tflite', recursive=True):
    files.download(tflite)


## ขั้น 9 — ส่งไฟล์ให้ผม

ได้ `wakeword.onnx` แล้วให้บอกผม พร้อมสามอย่างนี้:

1. ตารางผลวัดจากขั้น 7
2. threshold ที่เลือก
3. ขนาดไฟล์

ผมจะต่อเข้าแอป Android ให้ (ดู `WAKEWORD.md` หัวข้อการนำโมเดลเข้าแอป)
แล้วเราวัดจริงบน A07 ตามขั้นตอนที่เตรียมไว้

**ยังไม่ต้องอัปโหลดไฟล์เข้า repo** — โมเดลเป็นไฟล์ binary ควรอยู่ในเครื่อง
ไม่ใช่ใน git
